# Eval 7 and Eval 8 Comparison Notebook

<details open>
<summary><strong>Top-Level Eval Map (click to collapse)</strong></summary>

This notebook has two top-level, self-contained evals:
1. Eval 7: full-dataset gated vs no-gate comparison (includes setup, train, validation, and artifacts).
2. Eval 8: optional small discrete sweep over routing/gate hyperparameters.

Run Eval 7 end-to-end first, then run Eval 8 if you want additional ablations.

</details>

Objective:
- Keep Eval 7 flow aligned with utility patterns from logic_hf.
- Align head output dimension with ProofWriter labels.
- Use the full ProofWriter dataset (all available depths).
- Compare gated logic stream vs no-gate stream.
- Save resumable checkpoints so training can continue later.

Experiment matrix (Eval 7):
- `gated`: `use_no_gate_stream=False`
- `no_gate`: `use_no_gate_stream=True`, `no_gate_match_logic_params=True`

## Architecture Diagram (ASCII)

```text
Input Text
   |
   v
Tokenizer + Batch Collator
   |
   v
Backbone Transformer (Llama 3.1 8B)
   |  hidden states per layer [B, S, H]
   v
Logic Projection (H -> L)
   |  projected states per layer [B, S, L]
   v
+---------------------------------------------------------------+
| Logic Stream (aligned with transformer layers)                |
|                                                               |
|  For each layer i:                                            |
|    token_logic_i [B, S, L]                                    |
|        |                                                      |
|        +--> (optional) Pre-Routing MLP (L -> L -> L -> L)    |
|        |          hyperparam: PRE_ROUTING_MLP                |
|        v                                                      |
|    RoutingModule: Linear(L->(3G)) -> /temperature -> softmax  |
|        hyperparams: NUM_GATES=G (per type),                  |
|                     ROUTING_TEMPERATURE, ROUTING_TOP_K        |
|        |                                                      |
|        v                                                      |
|    routing_weights [B, S, 3G]                                 |
|        |                                                      |
|        +--> gate_inputs via einsum("bsg,bsl->bgl") [B, 3G, L] |
|                   |                                           |
|                   v                                           |
|            gate_scalar + sigmoid -> gate_values [B, 3G]       |
|                   |                                           |
|                   v                                           |
|        Split into banks (G each): AND | OR | NOT              |
|                   |                                           |
|                   v                                           |
|        Gate Composition (operator + capacity)                 |
|          - each gate consumes CAPACITY inputs                 |
|          - AND: 1 output per gate -> [B, G]                   |
|          - OR:  1 output per gate -> [B, G]                   |
|          - NOT: CAPACITY outputs per gate -> [B, G*CAPACITY]  |
|          - all mode concatenates [AND, OR, NOT]               |
|                   |                                           |
|                   v                                           |
|            logic_update: Linear(G*(CAPACITY+2) -> L) + norm   |
|                   |                                           |
|                   v                                           |
|             logic_state [B, L] (final after last layer)       |
+---------------------------------------------------------------+

Final backbone hidden [B, S, H]
   +
Broadcast logic_state [B, S, L]
   |
   v
FusionMLP -> fused_hidden [B, S, H]
   |
   v
Pooling (causal: last token)
   |
   v
Pre-head norm -> Task head -> logits [B, num_labels]

Variants in this notebook:
- gated: use_no_gate_stream=False (routing + gates enabled)
- no_gate: use_no_gate_stream=True (control path without routing/gates)
```


# Eval 7) Full-Dataset Gated vs No-Gate Comparison

<details>
<summary>Setup note</summary>

If needed, run this once in VS Code terminal:

```bash
python -m venv .venv
.\\.venv\\Scripts\\activate
python -m pip install --upgrade pip ipykernel
python -m ipykernel install --user --name ai535project --display-name "Python (ai535project)"
```

Then select the `Python (ai535project)` kernel.

</details>

## Eval 7.1 Install and Verify Dependencies

<details>
<summary>Execution note</summary>

Run the next 2 code cells in order.

</details>

In [ ]:
import sys

# Install only what is missing for this workflow.
%pip install -q pyyaml datasets transformers torch matplotlib huggingface_hub

In [ ]:
import platform
import torch
import transformers
import datasets
import yaml

print('python:', platform.python_version())
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
# Lightweight Hugging Face access preflight for Colab/local runs.
import importlib.util
import os
from datasets import load_dataset_builder
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

IN_COLAB = importlib.util.find_spec("google.colab") is not None

MODEL_CANDIDATES = [
    "meta-llama/Meta-Llama-3.1-8B",
    "meta-llama/Llama-3.1-8B-Instruct",
]
DATASET_CHECKS = [
    ("tasksource/proofwriter", "default"),
]

hf_token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

print("HF token found:", bool(hf_token))
api = HfApi()
model_ok = {}
for model_id in MODEL_CANDIDATES:
    try:
        _ = api.model_info(repo_id=model_id, token=hf_token)
        model_ok[model_id] = True
        print(f"[OK] model access: {model_id}")
    except HfHubHTTPError as e:
        model_ok[model_id] = False
        code = getattr(e.response, "status_code", "unknown")
        print(f"[FAIL] model access: {model_id} -> HTTP {code}")
    except Exception as e:
        model_ok[model_id] = False
        print(f"[FAIL] model access: {model_id} -> {type(e).__name__}: {e}")

dataset_ok = {}
for ds_name, ds_cfg in DATASET_CHECKS:
    try:
        _ = load_dataset_builder(ds_name, ds_cfg) if ds_cfg else load_dataset_builder(ds_name)
        dataset_ok[(ds_name, ds_cfg)] = True
        suffix = f"/{ds_cfg}" if ds_cfg else ""
        print(f"[OK] dataset access: {ds_name}{suffix}")
    except Exception as e:
        dataset_ok[(ds_name, ds_cfg)] = False
        suffix = f"/{ds_cfg}" if ds_cfg else ""
        print(f"[FAIL] dataset access: {ds_name}{suffix} -> {type(e).__name__}: {e}")

LLAMA_PREFLIGHT_OK = all(model_ok.values()) and all(dataset_ok.values())
print("LLAMA_PREFLIGHT_OK =", LLAMA_PREFLIGHT_OK)

In [ ]:
# -- Colab setup for private repo clone/pull (safe token handling) ---------------
# Skip this cell if running locally in VS Code.
import os
import sys

if "google.colab" in sys.modules:
    from getpass import getpass
    from google.colab import userdata

    REPO_URL = "github.com/Ancorro/AI535project.git"
    REPO_DIR = "/content/AI535project"
    TARGET_BRANCH = "V2"

    # Prefer Colab Secrets first, then env var, then prompt as last resort.
    gh_token = (
        userdata.get("GITHUB_TOKEN")
        or userdata.get("GH_TOKEN")
        or os.getenv("GITHUB_TOKEN")
    )
    if not gh_token:
        gh_token = getpass("GitHub PAT (repo read access): ")

    auth_url = f"https://{gh_token}@{REPO_URL}"
    if os.path.isdir(REPO_DIR):
        print(f"Repo exists, switching/updating branch {TARGET_BRANCH}...")
        ! git -C {REPO_DIR} fetch origin {TARGET_BRANCH}
        ! git -C {REPO_DIR} checkout {TARGET_BRANCH}
        ! git -C {REPO_DIR} pull origin {TARGET_BRANCH}
    else:
        print(f"Repo not found, cloning branch {TARGET_BRANCH}...")
        ! git clone --branch {TARGET_BRANCH} --single-branch {auth_url} {REPO_DIR}

    del gh_token, auth_url

    %cd /content/AI535project/logic

## Eval 7.2 HYPERPARAMETERS, Reproducibility and Token Resolution

<details>
<summary>Execution note</summary>

Run the next 2 code cells to set hyperparameters, repo root, and HF token resolution.

</details>

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import torch

SEED = 42
USE_ALL_DEPTHS = True
NODE_ID = 7
BATCH_SIZE = 100
LEARNING_RATE = 1e-4
EPOCHS = 5
EVALS_PER_EPOCH = 40  # validate this many times per epoch (1 = epoch-end only)

# Checkpoint saving controls.
SAVE_CHECKPOINTS = False  # master toggle for all checkpoint artifact writing (checkpoint_last.pt and checkpoint.pt)
SAVE_CHECKPOINT_EVERY_EPOCH = True  # if False, save only once at end of training

# LoRA control at notebook level.
USE_LORA = False  # set True to enable LoRA adapters for this run

# Backbone freeze control at notebook level.
FREEZE_BACKBONE = False  # True: freeze Llama backbone, False: finetune backbone

# Fusion control at notebook level.
FUSION_ALPHA_INIT = 0.3  # initial residual fusion strength
LEARN_FUSION_ALPHA = False  # True: train alpha, False: keep alpha fixed at init

# W&B logging mode control.
# 'offline' is faster between variants; sync later with: wandb sync wandb/offline-run-*
WANDB_MODE = 'online'  # one of: online, offline

MODEL_NAME = 'meta-llama/Meta-Llama-3.1-8B'
PROOFWRITER_CONFIG = 'default'
K_FOLDS = 10
K_FOLD_INDEX = 0

# Scheduler controls (easy notebook-level controls).
SCHEDULER_TYPE = 'warmup_cosine'  # one of: warmup_constant, warmup_cyclic, warmup_cosine
SCHEDULER_ETA_MIN_RATIO = 0.1  # used by warmup_cosine only

# Routing hyperparameters (easy notebook-level controls).
ROUTING_TOP_K = 'all' # int for pruning, or 'all' for no pruning during inference
ROUTING_TEMPERATURE = 1.0

# Logic gate hyperparameters.
NUM_GATES = 16  # G gates per operator bank (AND/OR/NOT), total routed gates = 3 * G
GATE_OPERATOR = 'all'  # one of: all, and, or, not
GATE_MODE = 'soft'  # one of: soft, ste_binary
GATE_CAPACITY = 2  # each gate consumes CAPACITY inputs
PRE_ROUTING_MLP = True  # logic_dim -> logic_dim MLP before routing
SKIP_PROJECTION_WHEN_DIMS_MATCH = True  # if logic_dim == Llama hidden_size, use identity (no learned map)

# Resolve repository root from current working directory (works for VS Code and Colab).
if 'REPO_ROOT' not in globals():
    cwd = Path.cwd().resolve()
    repo_candidate = None
    for p in [cwd, *cwd.parents]:
        if (p / 'train' / 'train.py').exists() and (p / 'logic').exists():
            repo_candidate = p
            break
    if repo_candidate is None:
        raise FileNotFoundError('Could not locate REPO_ROOT containing train/train.py and logic/.')
    REPO_ROOT = repo_candidate

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('use all depths:', USE_ALL_DEPTHS)
print('k_folds:', K_FOLDS)
print('k_fold_index:', K_FOLD_INDEX)
print('scheduler_type:', SCHEDULER_TYPE)
print('scheduler_eta_min_ratio:', SCHEDULER_ETA_MIN_RATIO)
print('routing_top_k:', ROUTING_TOP_K)
print('routing_temperature:', ROUTING_TEMPERATURE)
print('num_gates_per_type_G:', NUM_GATES)
print('total_routed_gates_per_layer_3G:', 3 * NUM_GATES)
print('gate_operator:', GATE_OPERATOR)
print('gate_mode:', GATE_MODE)
print('gate_capacity:', GATE_CAPACITY)
print('all_mode_logic_update_in_dim:', NUM_GATES * (GATE_CAPACITY + 2))
print('pre_routing_mlp:', PRE_ROUTING_MLP)
print('skip_projection_when_dims_match:', SKIP_PROJECTION_WHEN_DIMS_MATCH)
print('evals_per_epoch:', EVALS_PER_EPOCH)
print('save_checkpoints:', SAVE_CHECKPOINTS)
print('save_checkpoint_every_epoch:', SAVE_CHECKPOINT_EVERY_EPOCH)
print('use_lora:', USE_LORA)
print('freeze_backbone:', FREEZE_BACKBONE)
print('fusion_alpha_init:', FUSION_ALPHA_INIT)
print('learn_fusion_alpha:', LEARN_FUSION_ALPHA)
print('wandb_mode:', WANDB_MODE)
print('repo root:', REPO_ROOT)

In [ ]:
# logic_hf-style token resolution: Colab secrets first, then env vars.
HF_TOKEN_RESOLVED = None
if 'IN_COLAB' in globals() and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN_RESOLVED = userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN')
    except Exception:
        HF_TOKEN_RESOLVED = None

if not HF_TOKEN_RESOLVED:
    HF_TOKEN_RESOLVED = os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACE_TOKEN')

if HF_TOKEN_RESOLVED:
    os.environ['HF_TOKEN'] = HF_TOKEN_RESOLVED
    os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN_RESOLVED

print('HF token resolved:', bool(HF_TOKEN_RESOLVED))

In [ ]:
import copy
import json
import os
import subprocess
import sys
from collections import Counter
from datetime import datetime

import yaml
from datasets import load_dataset

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from logic.core.data_utils import build_dataloader
from train.train import _load_config

In [ ]:
# Shared helpers (ported from logic_hf-style workflow) to keep this notebook self-contained.
from pathlib import Path
from typing import Any
from collections import deque

def make_json_safe(obj: Any):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    return obj

def write_variant_config(variant_name: str, overrides: dict) -> Path:
    cfg = _build_node7_config(
        base_cfg,
        variant_name=variant_name,
        model_name=MODEL_NAME,
        num_labels=inferred_num_labels,
    )

    model_overrides = {
        'num_gates',
        'gate_operator',
        'gate_mode',
        'gate_capacity',
        'routing_top_k',
        'routing_temperature',
        'pre_routing_mlp',
    }
    train_overrides = {'learning_rate', 'epochs', 'batch_size'}

    for k, v in overrides.items():
        if k in model_overrides:
            cfg['model'][k] = v
        elif k in train_overrides:
            cfg['train'][k] = v
        else:
            cfg[k] = v

    cfg.setdefault('wandb', {})
    cfg['wandb']['group'] = NODE7_WANDB_GROUP
    cfg['wandb']['run_name'] = f"node8_{variant_name}"

    out_path = EXPERIMENT_CONFIG_DIR / f"node8_{variant_name}.yaml"
    with open(out_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out_path

def _tail_text_file(path: Path, max_lines: int = 80) -> str:
    tail = deque(maxlen=max_lines)
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            tail.append(line.rstrip('\n'))
    return '\n'.join(tail)

def run_train_for_config(cfg_path: Path) -> Path:
    train_script = REPO_ROOT / 'train' / 'train.py'
    cmd = [sys.executable, str(train_script), '--config_path', str(cfg_path)]

    run_log_path = REPO_ROOT / 'runs' / f"{cfg_path.stem}_train.log"
    with open(run_log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.run(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
            env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        )

    print(_tail_text_file(run_log_path, max_lines=80))
    if proc.returncode != 0:
        raise RuntimeError(f"Training failed for config: {cfg_path}; see {run_log_path}")

    run_dirs = sorted((REPO_ROOT / 'runs').glob('run_*'), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError('No run directories found after training.')
    return run_dirs[-1]

def summarize_run_dir(run_dir: Path) -> dict:
    summary = {
        'run_dir': str(run_dir),
        'checkpoint_last_exists': (run_dir / 'checkpoint_last.pt').exists(),
        'checkpoint_model_exists': (run_dir / 'checkpoint.pt').exists(),
    }
    metrics = {}
    for name in ['metrics.json', 'eval_metrics.json', 'train_metrics.json']:
        p = run_dir / name
        if p.exists():
            try:
                with open(p, 'r', encoding='utf-8') as f:
                    metrics[name] = json.load(f)
            except Exception as e:
                metrics[name] = {'error': f"{type(e).__name__}: {e}"}
    summary['metrics_files'] = metrics
    return summary

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / 'configs' / 'config.yaml'
EXPERIMENT_CONFIG_DIR = REPO_ROOT / 'configs' / 'experiments'
EXPERIMENT_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

import os
from datetime import datetime

# Fallback for out-of-order execution when import cell was skipped.
if '_load_config' not in globals():
    from pathlib import Path
    import yaml
    def _load_config(config_path):
        path = Path(config_path)
        if not path.exists():
            raise FileNotFoundError(f"Config not found: {path}")
        with open(path, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f)

base_cfg = _load_config(BASE_CONFIG_PATH)
base_cfg.setdefault('model', {})
base_cfg.setdefault('train', {})
base_cfg.setdefault('data', {})
base_cfg.setdefault('wandb', {})

# Mirror logic_hf node pattern: per-node wandb grouping.
NODE7_WANDB_GROUP = f"node7_nogate_compare_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

WANDB_API_KEY = userdata.get('WANDB_API_KEY') if 'userdata' in globals() else os.getenv('WANDB_API_KEY')
if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    print('WANDB_API_KEY resolved; wandb enabled for Eval 7 runs.')
else:
    print('WANDB_API_KEY not found. Continuing with configured wandb mode.')

base_cfg['wandb']['mode'] = WANDB_MODE
base_cfg['model']['model_name'] = MODEL_NAME
base_cfg['model']['freeze_backbone'] = bool(FREEZE_BACKBONE)
base_cfg['model']['alpha_init'] = float(FUSION_ALPHA_INIT)
base_cfg['model']['learn_fusion_alpha'] = bool(LEARN_FUSION_ALPHA)
base_cfg['train']['batch_size'] = BATCH_SIZE
base_cfg['train']['epochs'] = EPOCHS
base_cfg['train']['evals_per_epoch'] = EVALS_PER_EPOCH
base_cfg['train']['save_checkpoints'] = SAVE_CHECKPOINTS
base_cfg['train']['save_checkpoint_every_epoch'] = SAVE_CHECKPOINT_EVERY_EPOCH
base_cfg['train']['learning_rate'] = LEARNING_RATE
base_cfg['train']['seed'] = SEED

# LoRA toggle: preserve existing LoRA hyperparameters, only flip enabled.
base_cfg['model'].setdefault('lora', {})
base_cfg['model']['lora']['enabled'] = bool(USE_LORA)

# Use proofwriter loader path in data_utils (handles label normalization for answer strings).
base_cfg['data']['dataset'] = 'proofwriter'
base_cfg['data']['config'] = PROOFWRITER_CONFIG
base_cfg['data']['depth'] = 'all'
base_cfg['data']['max_length'] = 256

print('Loaded base config from', BASE_CONFIG_PATH)
print('Eval 7 wandb group:', NODE7_WANDB_GROUP)
print('Configured dataset:', base_cfg['data']['dataset'], '/', base_cfg['data']['config'])
print('Configured depth:', base_cfg['data']['depth'])
print('Configured evals_per_epoch:', base_cfg['train']['evals_per_epoch'])
print('Configured save_checkpoints:', base_cfg['train']['save_checkpoints'])
print('Configured save_checkpoint_every_epoch:', base_cfg['train']['save_checkpoint_every_epoch'])
print('Configured use_lora:', base_cfg['model']['lora']['enabled'])
print('Configured freeze_backbone:', base_cfg['model']['freeze_backbone'])
print('Configured fusion alpha init:', base_cfg['model']['alpha_init'])
print('Configured learn_fusion_alpha:', base_cfg['model']['learn_fusion_alpha'])
print('Configured wandb mode:', base_cfg['wandb']['mode'])

In [ ]:
def _extract_label(ex):
    if 'answer' in ex and ex['answer'] is not None:
        val = ex['answer']
        if isinstance(val, bool):
            return 1 if val else 0
        if isinstance(val, (int, float)) and not isinstance(val, bool):
            iv = int(val)
            if iv in (0, 1, 2):
                return iv
            if iv in (1, 2, 3):
                return iv - 1
        txt = str(val).strip().lower()
        if txt in {'false', 'no', '0', 'contradiction', 'refuted', 'refutes'}:
            return 0
        if txt in {'true', 'yes', '1', 'entailment', 'entailed', 'supports', 'provable'}:
            return 1
        if txt in {'unknown', 'uncertain', 'both', 'indeterminate', 'neither', 'inconclusive'}:
            return 2
        raise ValueError(f'Unrecognized ProofWriter answer label: {val!r}')
    if 'label' in ex and ex['label'] is not None:
        return int(ex['label'])
    raise KeyError('Expected `answer` or `label` in example')

def _extract_depth(ex):
    for key in ('depth', 'proof_depth', 'metadata'):
        if key in ex and ex[key] is not None:
            value = ex[key]
            if isinstance(value, int):
                return value
            text = str(value)
            digits = ''.join(ch for ch in text if ch.isdigit())
            if digits:
                return int(digits)
    return None

def inspect_proofwriter_depths(config_name=PROOFWRITER_CONFIG, split='train'):
    ds = load_dataset('tasksource/proofwriter', config_name, split=split)
    labels = [_extract_label(x) for x in ds]
    depths = [_extract_depth(x) for x in ds]
    depth_counter = Counter([d for d in depths if d is not None])
    return ds, labels, depths, depth_counter

raw_train_ds, raw_train_labels, raw_train_depths, depth_counts = inspect_proofwriter_depths(split='train')
print('train size (all depths):', len(raw_train_ds))
print('label counts:', Counter(raw_train_labels))
print('depth counts:', dict(sorted(depth_counts.items())))

In [ ]:
unique_labels = sorted(set(raw_train_labels))
inferred_num_labels = len(unique_labels)

print('unique labels:', unique_labels)
print('inferred_num_labels:', inferred_num_labels)

if inferred_num_labels != 3:
    raise ValueError(
        f'Expected ProofWriter to expose exactly 3 labels, got {inferred_num_labels} with labels {unique_labels}'
    )

base_cfg['model']['num_labels'] = inferred_num_labels
print('Configured model.num_labels =', base_cfg['model']['num_labels'])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from datasets import load_dataset

def _resolve_eval_split(cfg):
    dataset_name = str(cfg['data']['dataset']).lower()
    if dataset_name == 'proofwriter':
        pw_cfg = cfg['data'].get('config')
        try:
            available = list(load_dataset('tasksource/proofwriter', pw_cfg).keys()) if pw_cfg else list(load_dataset('tasksource/proofwriter').keys())
        except Exception:
            available = list(load_dataset('longface/ProofWriter').keys())
    else:
        ds_name = cfg['data']['dataset']
        ds_cfg = cfg['data'].get('config')
        available = list(load_dataset(ds_name, ds_cfg).keys()) if ds_cfg else list(load_dataset(ds_name).keys())

    preference = ['validation', 'val', 'dev', 'test', 'train']
    for split in preference:
        if split in available:
            return split, available
    raise ValueError(f'No usable evaluation split found. Available splits: {available}')

EVAL_SPLIT, available_splits = _resolve_eval_split(base_cfg)

train_loader, _ = build_dataloader(base_cfg, split='train')
if EVAL_SPLIT == 'train':
    from torch.utils.data import DataLoader, Subset
    K_FOLDS = int(globals().get('K_FOLDS', 5))
    K_FOLD_INDEX = int(globals().get('K_FOLD_INDEX', 0))
    if K_FOLDS < 2:
        raise ValueError(f'K_FOLDS must be >= 2, got {K_FOLDS}')
    if K_FOLD_INDEX < 0 or K_FOLD_INDEX >= K_FOLDS:
        raise ValueError(f'K_FOLD_INDEX must be in [0, {K_FOLDS - 1}], got {K_FOLD_INDEX}')

    base_dataset = train_loader.dataset
    n_total = len(base_dataset)
    if n_total < K_FOLDS:
        raise ValueError(f'Dataset size ({n_total}) is smaller than K_FOLDS ({K_FOLDS})')

    indices = list(range(n_total))
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)

    fold_size = n_total // K_FOLDS
    remainder = n_total % K_FOLDS
    start = K_FOLD_INDEX * fold_size + min(K_FOLD_INDEX, remainder)
    end = start + fold_size + (1 if K_FOLD_INDEX < remainder else 0)
    val_idx = indices[start:end]
    train_idx = indices[:start] + indices[end:]

    collate_fn = train_loader.collate_fn
    train_loader = DataLoader(Subset(base_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(Subset(base_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    EVAL_SPLIT = f'kfold:{K_FOLD_INDEX}/{K_FOLDS}'
else:
    val_loader, _ = build_dataloader(base_cfg, split=EVAL_SPLIT)

print('train batches:', len(train_loader))
print('val batches:', len(val_loader))
print('available splits:', available_splits)
print('eval split:', EVAL_SPLIT)
print('device:', device)

In [ ]:
def _build_node7_config(base_cfg: dict, variant_name: str, model_name: str, num_labels: int) -> dict:
    cfg = copy.deepcopy(base_cfg)
    cfg.setdefault('model', {})
    cfg.setdefault('train', {})
    cfg.setdefault('data', {})
    cfg.setdefault('wandb', {})

    cfg['model']['model_name'] = model_name
    cfg['model']['num_labels'] = int(num_labels)
    cfg['model']['use_logic_stream'] = True
    cfg['model']['stream_logic_during_backbone'] = True
    cfg['model']['freeze_backbone'] = bool(cfg['model'].get('freeze_backbone', True))

    # Scheduler controls exposed at notebook level.
    cfg['train']['scheduler'] = {'type': str(SCHEDULER_TYPE)}
    if SCHEDULER_TYPE == 'warmup_cosine':
        cfg['train']['scheduler']['eta_min_ratio'] = float(SCHEDULER_ETA_MIN_RATIO)

    # K-fold fallback for datasets without validation splits.
    cfg['train']['kfold'] = {
        'enabled': True,
        'n_splits': int(K_FOLDS),
        'fold_index': int(K_FOLD_INDEX),
    }

    # Routing controls exposed at notebook level.
    if ROUTING_TOP_K == 'all':
        cfg['model']['routing_top_k'] = 'all'
    else:
        cfg['model']['routing_top_k'] = int(ROUTING_TOP_K)
    cfg['model']['routing_temperature'] = float(ROUTING_TEMPERATURE)

    # Gate-count and gate-capacity are explicit hyperparameters.
    cfg['model']['num_gates'] = int(NUM_GATES)
    cfg['model']['gate_operator'] = str(GATE_OPERATOR)
    cfg['model']['gate_mode'] = str(GATE_MODE)
    cfg['model']['gate_capacity'] = int(GATE_CAPACITY)
    cfg['model']['pre_routing_mlp'] = bool(PRE_ROUTING_MLP)
    cfg['model']['skip_projection_when_dims_match'] = bool(SKIP_PROJECTION_WHEN_DIMS_MATCH)

    if variant_name == 'gated':
        cfg['model']['use_no_gate_stream'] = False
    elif variant_name == 'no_gate':
        cfg['model']['use_no_gate_stream'] = True
        cfg['model']['no_gate_update_type'] = 'mlp'
        cfg['model']['no_gate_match_logic_params'] = True
    else:
        raise ValueError(f'Unknown variant: {variant_name}')

    cfg['train']['batch_size'] = int(BATCH_SIZE)
    cfg['train']['learning_rate'] = float(LEARNING_RATE)
    cfg['train']['epochs'] = int(EPOCHS)
    cfg['train']['evals_per_epoch'] = int(EVALS_PER_EPOCH)
    cfg['train']['save_checkpoints'] = bool(SAVE_CHECKPOINTS)
    cfg['train']['save_checkpoint_every_epoch'] = bool(SAVE_CHECKPOINT_EVERY_EPOCH)
    cfg['train']['seed'] = int(SEED)

    cfg['model'].setdefault('lora', {})
    cfg['model']['lora']['enabled'] = bool(USE_LORA)

    cfg['model']['alpha_init'] = float(FUSION_ALPHA_INIT)
    cfg['model']['learn_fusion_alpha'] = bool(LEARN_FUSION_ALPHA)

    cfg['data']['dataset'] = 'proofwriter'
    cfg['data']['config'] = PROOFWRITER_CONFIG
    cfg['data']['depth'] = 'all'

    cfg['wandb']['mode'] = WANDB_MODE
    cfg['wandb']['group'] = NODE7_WANDB_GROUP
    cfg['wandb']['run_name'] = f'node{NODE_ID}_{variant_name}'
    return cfg


VARIANTS = ['gated', 'no_gate']
variant_config_paths = {}
for name in VARIANTS:
    cfg = _build_node7_config(base_cfg, name, model_name=MODEL_NAME, num_labels=inferred_num_labels)
    out_path = EXPERIMENT_CONFIG_DIR / f'node{NODE_ID}_{name}.yaml'
    with open(out_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    variant_config_paths[name] = out_path

print('Wrote Eval 7 configs:')
for k, v in variant_config_paths.items():
    print(f'  - {k}: {v}')

## Eval 7.3 TRAIN

In [ ]:
# Train each Eval 7 variant using a streaming log flow (avoids huge in-memory stdout buffers).
variant_run_dirs = {}
train_script = REPO_ROOT / 'train' / 'train.py'

for name in VARIANTS:
    cfg_path = variant_config_paths[name]
    print(f"\n=== Node {NODE_ID} training: {name} ===")
    cmd = [sys.executable, str(train_script), '--config_path', str(cfg_path)]

    # Write subprocess output directly to disk to reduce notebook lag after each run.
    run_log_path = REPO_ROOT / 'runs' / f'node{NODE_ID}_{name}_train.log'
    with open(run_log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.run(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
            env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        )

    print(_tail_text_file(run_log_path, max_lines=80))
    if proc.returncode != 0:
        raise RuntimeError(f'Training failed for {name}; see {run_log_path}')

    run_dirs = sorted((REPO_ROOT / 'runs').glob('run_*'), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise RuntimeError('No run directories found after training.')
    variant_run_dirs[name] = run_dirs[-1]

node7_report = {
    'node_id': NODE_ID,
    'wandb_group': NODE7_WANDB_GROUP,
    'notebook_hyperparameters': {
        'seed': SEED,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
        'evals_per_epoch': EVALS_PER_EPOCH,
        'save_checkpoints': SAVE_CHECKPOINTS,
        'save_checkpoint_every_epoch': SAVE_CHECKPOINT_EVERY_EPOCH,
        'use_lora': USE_LORA,
        'freeze_backbone': FREEZE_BACKBONE,
        'fusion_alpha_init': FUSION_ALPHA_INIT,
        'learn_fusion_alpha': LEARN_FUSION_ALPHA,
        'wandb_mode': WANDB_MODE,
        'model_name': MODEL_NAME,
        'proofwriter_config': PROOFWRITER_CONFIG,
        'scheduler_type': SCHEDULER_TYPE,
        'scheduler_eta_min_ratio': SCHEDULER_ETA_MIN_RATIO,
        'routing_top_k': ROUTING_TOP_K,
        'routing_temperature': ROUTING_TEMPERATURE,
        'num_gates': NUM_GATES,
        'gate_operator': GATE_OPERATOR,
        'gate_mode': GATE_MODE,
        'gate_capacity': GATE_CAPACITY,
        'pre_routing_mlp': PRE_ROUTING_MLP,
    },
    'variants': {},
}

for name in VARIANTS:
    run_dir = variant_run_dirs[name]
    cfg_path = variant_config_paths[name]
    run_cfg = _load_config(cfg_path)

    run_config_path = run_dir / 'config.json'
    run_cfg_logged = {}
    if run_config_path.exists():
        with open(run_config_path, 'r', encoding='utf-8') as f:
            run_cfg_logged = json.load(f)

    # Save a dedicated hyperparameter snapshot per run for quick inspection.
    hparams_snapshot_path = run_dir / f'node{NODE_ID}_{name}_hparams_snapshot.json'
    with open(hparams_snapshot_path, 'w', encoding='utf-8') as f:
        json.dump(
            {
                'variant': name,
                'requested_config_path': str(cfg_path),
                'requested_config': run_cfg,
                'logged_run_config_path': str(run_config_path),
                'logged_run_config': run_cfg_logged,
                'notebook_hyperparameters': node7_report['notebook_hyperparameters'],
            },
            f,
            indent=2,
        )

    node7_report['variants'][name] = {
        'config_path': str(cfg_path),
        'run_dir': str(run_dir),
        'run_config_path': str(run_config_path),
        'hparams_snapshot_path': str(hparams_snapshot_path),
        'checkpoint_last': str(run_dir / 'checkpoint_last.pt'),
        'checkpoint_model': str(run_dir / 'checkpoint.pt'),
        'wandb_run_name': run_cfg.get('wandb', {}).get('run_name'),
        'wandb_mode': run_cfg.get('wandb', {}).get('mode'),
        'save_checkpoints': run_cfg.get('train', {}).get('save_checkpoints'),
        'save_checkpoint_every_epoch': run_cfg.get('train', {}).get('save_checkpoint_every_epoch'),
        'use_lora': run_cfg.get('model', {}).get('lora', {}).get('enabled'),
        'freeze_backbone': run_cfg.get('model', {}).get('freeze_backbone'),
        'fusion_alpha_init': run_cfg.get('model', {}).get('alpha_init'),
        'learn_fusion_alpha': run_cfg.get('model', {}).get('learn_fusion_alpha'),
    }

summary_path = REPO_ROOT / 'runs' / f'node{NODE_ID}_comparison_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(node7_report, f, indent=2)

print(f"\nNode {NODE_ID} run directories:")
for k, v in variant_run_dirs.items():
    print(f'  - {k}: {v}')
print('Saved summary to', summary_path)

## Eval 7.4 Validation Checks

<details>
<summary>Execution note</summary>

Run the next code cell to confirm batch and label integrity.

</details>

In [ ]:
sample_batch = next(iter(train_loader))
for key in ('input_ids', 'attention_mask', 'labels'):
    assert key in sample_batch, f'Missing batch key: {key}'

assert sample_batch['input_ids'].shape == sample_batch['attention_mask'].shape
assert sample_batch['labels'].ndim == 1
assert sample_batch['labels'].min().item() >= 0
assert sample_batch['labels'].max().item() < base_cfg['model']['num_labels']
print('Batch assertions passed.')

## Eval 7.5 Artifacts and Checkpoint Resume

<details>
<summary>Execution note</summary>

Run the next 2 code cells to inspect summaries and test checkpoint resume.

</details>

In [ ]:
summary_path = REPO_ROOT / 'runs' / f'node{NODE_ID}_comparison_summary.json'
assert summary_path.exists(), f'Missing summary file: {summary_path}'

with open(summary_path, 'r', encoding='utf-8') as f:
    loaded_summary = json.load(f)

print('Loaded summary keys:', sorted(loaded_summary.keys()))
print('Notebook hyperparameters:', loaded_summary.get('notebook_hyperparameters', {}))
print('Saved artifacts from summary:')
for variant_name, info in loaded_summary['variants'].items():
    print(f"- {variant_name}")
    print('  run_dir:', info['run_dir'])
    print('  run_config_path:', info.get('run_config_path'))
    print('  hparams_snapshot_path:', info.get('hparams_snapshot_path'))
    print('  checkpoint_last:', info['checkpoint_last'])
    print('  checkpoint_model:', info['checkpoint_model'])

In [ ]:
def resume_variant_from_checkpoint(variant_name):
    if variant_name not in loaded_summary['variants']:
        raise KeyError(f'Unknown variant: {variant_name}')

    ckpt_path = Path(loaded_summary['variants'][variant_name]['checkpoint_last'])
    assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'

    ckpt = torch.load(ckpt_path, map_location='cpu')
    print('resumed variant:', variant_name)
    print('checkpoint path:', ckpt_path)
    print('epoch index:', ckpt.get('epoch'))
    print('global_step:', ckpt.get('global_step'))
    return ckpt

_ = resume_variant_from_checkpoint('gated')

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Build/load comparison report in a logic_hf-compatible schema.
if 'comparison_report' not in globals():
    chart_summary_path = REPO_ROOT / 'runs' / f'node{NODE_ID}_chart_comparison_summary.json'
    if chart_summary_path.exists():
        with open(chart_summary_path, 'r', encoding='utf-8') as f:
            comparison_report = json.load(f)
    else:
        from scripts.test_logic_vs_llama31 import run_benchmark

        comparison_report = {
            'variants': {},
            'settings': {
                'datasets': ['proofwriter'],
                'split': ('train' if str(globals().get('EVAL_SPLIT', 'validation')).startswith('kfold:') else globals().get('EVAL_SPLIT', 'validation')),
                'max_samples': 64,
                'batch_size': 8,
                'max_length': 256,
                'llama_model_name': 'meta-llama/Llama-3.1-8B-Instruct',
                'skip_llama': False,
            },
        }

        for name in VARIANTS:
            run_dir = variant_run_dirs[name]
            print(f"\n=== Node {NODE_ID} benchmark: {name} ===")
            report = run_benchmark(
                run_dir=run_dir,
                datasets=comparison_report['settings']['datasets'],
                split=comparison_report['settings']['split'],
                max_samples=comparison_report['settings']['max_samples'],
                batch_size=comparison_report['settings']['batch_size'],
                max_length=comparison_report['settings']['max_length'],
                llama_model_name=comparison_report['settings']['llama_model_name'],
                skip_llama=comparison_report['settings']['skip_llama'],
                output=REPO_ROOT / 'runs' / f'logic_vs_llama31_node{NODE_ID}_{name}.json',
            )
            comparison_report['variants'][name] = report

        with open(chart_summary_path, 'w', encoding='utf-8') as f:
            json.dump(comparison_report, f, indent=2)

_cmp = comparison_report
variants = list(_cmp.get('variants', {}).keys())
if not variants:
    raise RuntimeError("No variants found in comparison_report['variants']")

dataset_keys = sorted({
    ds
    for v in variants
    for ds in _cmp['variants'].get(v, {}).get('datasets', {}).keys()
})
if not dataset_keys:
    raise RuntimeError('No dataset metrics found in comparison report')

proj_acc = np.full((len(variants), len(dataset_keys)), np.nan, dtype=float)
llama_acc = np.full((len(variants), len(dataset_keys)), np.nan, dtype=float)

for i, v in enumerate(variants):
    ds_block = _cmp['variants'].get(v, {}).get('datasets', {})
    for j, ds in enumerate(dataset_keys):
        d = ds_block.get(ds, {})
        proj_acc[i, j] = d.get('project_model', {}).get('accuracy', np.nan)
        llama_acc[i, j] = d.get('llama_3_1_8b', {}).get('accuracy', np.nan)

num_ds = len(dataset_keys)
fig, axes = plt.subplots(1, num_ds, figsize=(6 * num_ds, 4), squeeze=False)
axes = axes[0]

for j, ds in enumerate(dataset_keys):
    ax = axes[j]
    x = np.arange(len(variants))
    width = 0.35

    ax.bar(x - width / 2, proj_acc[:, j], width=width, label='Project', color='#2f6db3')
    if np.isfinite(llama_acc[:, j]).any():
        ax.bar(x + width / 2, llama_acc[:, j], width=width, label='Llama 3.1 8B', color='#c44e52')

    ax.set_title(f'Accuracy on {ds}')
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(variants, rotation=20, ha='right')
    ax.set_ylabel('Accuracy')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(loc='best')

plt.tight_layout()
plt.show()

# Heatmap: project minus llama.
delta = proj_acc - llama_acc
fig, ax = plt.subplots(figsize=(1.8 * len(dataset_keys) + 3, 0.7 * len(variants) + 2.5))
im = ax.imshow(delta, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)

ax.set_title('Project - Llama accuracy delta')
ax.set_xticks(np.arange(len(dataset_keys)))
ax.set_xticklabels(dataset_keys)
ax.set_yticks(np.arange(len(variants)))
ax.set_yticklabels(variants)
ax.set_xlabel('Dataset')
ax.set_ylabel('Variant')

for i in range(len(variants)):
    for j in range(len(dataset_keys)):
        val = delta[i, j]
        txt = 'nan' if not np.isfinite(val) else f'{val:+.3f}'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9, color='black')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Accuracy delta')
plt.tight_layout()
plt.show()

In [ ]:
import itertools
import random
from datetime import datetime
from scripts.test_logic_vs_llama31 import run_benchmark

# Eval 8: small discrete sweep over logic hyperparameters.
# Set to True to actually launch training runs.
RUN_NODE8_SWEEP = False
NODE8_MAX_RUNS = 6
NODE8_SAMPLE_SEED = 42

NODE8_SPACE = {
    "gate_operator": ["all", "and", "or", "not"],
    "gate_mode": ["soft", "ste_binary"],
    "routing_top_k": [4, 8, "all"],
    "gate_capacity": [2, 4],
    "num_gates": [16, 24],
}

keys = list(NODE8_SPACE.keys())
all_combos = [dict(zip(keys, vals)) for vals in itertools.product(*(NODE8_SPACE[k] for k in keys))]

rng = random.Random(NODE8_SAMPLE_SEED)
rng.shuffle(all_combos)
sampled = all_combos[: min(NODE8_MAX_RUNS, len(all_combos))]

print(f"Eval 8 candidate pool: {len(all_combos)}")
print(f"Eval 8 sampled runs:   {len(sampled)}")
for i, combo in enumerate(sampled, start=1):
    print(f"  {i:02d}: {combo}")

node8_report = {
    "node_id": 8,
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    "run_enabled": RUN_NODE8_SWEEP,
    "max_runs": NODE8_MAX_RUNS,
    "sample_seed": NODE8_SAMPLE_SEED,
    "search_space": NODE8_SPACE,
    "sampled": sampled,
    "results": [],
}

if RUN_NODE8_SWEEP:
    for i, combo in enumerate(sampled, start=1):
        variant_name = f"node8sweep_{i:02d}"
        overrides = {
            "run_name_suffix": f"node8_{i:02d}",
            "num_gates": combo["num_gates"],
            "gate_operator": combo["gate_operator"],
            "gate_mode": combo["gate_mode"],
            "gate_capacity": combo["gate_capacity"],
            "routing_top_k": combo["routing_top_k"],
        }
        cfg_path = write_variant_config(variant_name, overrides)
        print(f"\n=== Eval 8 sweep run {i}/{len(sampled)}: {variant_name} ===")
        run_dir = run_train_for_config(cfg_path)
        metrics = summarize_run_dir(run_dir)
        node8_report["results"].append(
            {
                "variant": variant_name,
                "overrides": overrides,
                "run_dir": str(run_dir),
                "metrics": metrics,
            }
        )

    print("\nEval 8 sweep finished.")
else:
    print("\nPreview mode only. Set RUN_NODE8_SWEEP = True to execute runs.")

node8_summary_path = REPO_ROOT / "runs" / "node8_sweep_summary.json"
with open(node8_summary_path, "w", encoding="utf-8") as f:
    json.dump(make_json_safe(node8_report), f, indent=2)

print("Saved Eval 8 summary to", node8_summary_path)

# Eval 8) Small Discrete Sweep (Optional)

<details>
<summary>Execution note</summary>

This eval runs a compact discrete sweep over gate and routing settings.

Suggested usage:
1. Leave `RUN_NODE8_SWEEP = False` to preview sampled configs.
2. Set `RUN_NODE8_SWEEP = True` to execute sampled runs.
3. Review `runs/node8_sweep_summary.json` for configs and metrics.

</details>

### Eval 8.1) Export/Re-run Checklist

<details>
<summary>Verification checklist</summary>

1. Restart kernel.
2. Run all cells.
3. Confirm artifacts are written under `runs/` including:
   - `node7_comparison_summary.json`
   - `node7_chart_comparison_summary.json`
   - `node8_sweep_summary.json`
4. Export from Colab/VS Code notebook menu to HTML or PDF.

</details>